In [5]:
import pandas as pd
import numpy as np
from modAL.models import ActiveLearner
from sklearn.model_selection import train_test_split

from tensorflow import keras
from tensorflow.keras.optimizers import *
from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.layers import Input, Activation, AveragePooling2D, Conv2DTranspose, ZeroPadding2D
from tensorflow.keras.layers import Dense , Dropout, Conv2D ,Flatten, Conv1D, BatchNormalization
from sklearn.metrics import mean_squared_error

from matplotlib import pyplot as plt
import seaborn as sns

font_size = 18

In [3]:
# load numpy raw data
with open('all_ae.npy', 'rb') as f:
    h2_ae = np.load(f)
    Pos = np.load(f)

h2_ae_shape = h2_ae.shape
h2_ae_1dcnn = h2_ae.reshape(h2_ae_shape[0], h2_ae_shape[1], -1)
h2_ae.shape, Pos.shape

((4979, 56, 37, 32), (4979, 3))

In [8]:
print("train-test splitting ae data...")
x_train_ae, x_test_ae, y_train_ae, y_test_ae = train_test_split(h2_ae, Pos, test_size=0.1, random_state=42)
x_train_ae_cnn, x_test_ae_cnn, y_train_ae_cnn, y_test_ae_cnn = train_test_split(h2_ae_1dcnn, Pos, test_size=0.1, random_state=42)

print("splitting training data and active learning pool...")
train_x_al, pool_x_al, train_y_al, pool_y_al = train_test_split(x_train_ae_cnn, y_train_ae_cnn, test_size=0.8, random_state=42)

h2_ae_shape = h2_ae.shape
h2_ae_1dcnn = h2_ae.reshape(h2_ae_shape[0], h2_ae_shape[1], -1)

train-test splitting ae data...
splitting training data and active learning pool...


In [7]:
print(f"Test samples: {len(y_test_ae_cnn)}, pool size: {len(pool_y_al)}")


Test samples: 498, pool size: 3585


In [9]:
def get_prediction_precision(regressor, x_test, y_test):
    y_pred = regressor.predict(x_test)
    mse = mean_squared_error(y_true=y_test, y_pred=y_pred)
    #print(f"current MSE: {mse}")
    return mse

def random_sampling(classifier, X_pool):
    n_samples = len(X_pool)
    query_idx = np.random.choice(range(n_samples))
    return [query_idx], X_pool[query_idx]

def cnn_model1(input_shape, opt=Adam(1e-3), dropout_rate=0.2):
    model = Sequential()
    model.add(Conv1D(16, 16, input_shape=input_shape, activation='relu'))
    model.add(Conv1D(32, 16, activation='relu'))
    model.add(Conv1D(32, 16, activation='relu'))
    model.add(Flatten())
    model.add(BatchNormalization())
    model.add(Dense(512))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dense(256))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(3))
    model.compile(loss='mean_squared_error', optimizer=opt)
    return model

print("constructing 1D-CNN model...")
lr = 5e-3
epochs = 50
decay_rate = lr / 50
model_ae = cnn_model1(train_x_al.shape[1:], opt=Adam(5e-3, decay=decay_rate))

constructing 1D-CNN model...


In [10]:
# 1. Fit on original training data
print(f"train_x_al:{train_x_al.shape}, train_y_al:{train_y_al.shape}")
model_ae.fit(train_x_al, train_y_al, epochs=1, validation_data=(
    x_test_ae_cnn, y_test_ae_cnn), verbose=1)
print(f"Initial round of training finished, MSE:{get_prediction_precision(model_ae, x_test_ae_cnn, y_test_ae_cnn)}")


train_x_al:(896, 56, 1184), train_y_al:(896, 3)
28/28 [==============================] - 2s 61ms/step - loss: 96417.9453 - val_loss: 270803.3438
Initial round of training finished, MSE:270803.30755262694
Begin active learning process...


In [ ]:
print("Begin active learning process...")
n_queries = 1700
mode = "random"
#queries_num = np.linspace(100, 3500, 35)
mse_dict = {}
all_chosen_samples = []
mode_dict = {
    "random": random_sampling,
}
print("Initialize active learner...")
regressor = ActiveLearner(
    estimator=model_ae,
    query_strategy=mode_dict[mode],
    X_training=train_x_al, y_training=train_y_al
)
print(f"active learning for {n_queries} epochs")

_chosen_x, _chosen_y = [], []
i = 0
xs, ys = [], []
pool_x_shape = pool_x_al.shape
while i <= n_queries:
    # get_prediction_precision(regressor, x_test_ae_cnn, y_test_ae_cnn)
    query_idx, query_instance = regressor.query(pool_x_al)
    _x, _y = pool_x_al[query_idx], pool_y_al[query_idx]
    pool_x_al = np.delete(pool_x_al, query_idx, axis=0)
    pool_y_al = np.delete(pool_y_al, query_idx, axis=0)
    #print(f"_x:{_x}, _y:{_y}")
    all_chosen_samples.append(_y)
    _chosen_x.append(_x)
    _chosen_y.append(_y)
    #_chosen_x = np.append(_chosen_x, _x)
    #_chosen_y = np.append(_chosen_y, _y)
    xs.append(_x)
    ys.append(_y)

    if i % 50 == 0 and i > 0:
        # batch mode AL teach
        regressor_copy = copy.copy(regressor)
        #regressor.teach(_chosen_x[0], _chosen_y[0], only_new=False)

        # _model = copy.copy(model_ae)
        #_new_x = np.concatenate([_new_x, _x])
        #_new_y = np.concatenate([_new_y, _y])
        xs_backup = xs.copy()
        ys_backup = ys.copy()
        xs = np.array(xs)
        ys = np.array(ys)
        x_shape, y_shape = xs.shape, ys.shape
        xs = xs.reshape(x_shape[0], -1, x_shape[-1])
        ys = ys.reshape(y_shape[0], -1)
        print(f"xs:{xs.shape}, ys:{ys.shape}")
        regressor.teach(xs, ys, only_new=False)
        #model_ae.fit(xs, ys, validation_data=(x_test_ae_cnn, y_test_ae_cnn), verbose=1)

        # _model.fit(_new_x, _new_y, epochs=10, validation_data=(
        #     x_test_ae_cnn, y_test_ae_cnn), verbose=1)
        mse = get_prediction_precision(regressor, x_test_ae_cnn, y_test_ae_cnn)
        #mse = get_prediction_precision(model_ae, x_test_ae_cnn, y_test_ae_cnn)

        if i <= 300 and mse > 150000:
            print(f"Encountered large MSE: {mse}")
            i = i - 1
            regressor = regressor_copy
            xs = xs_backup
            ys = ys_backup
            continue
        elif i > 300 and mse > 40000:
            print(f"Encountered large MSE: {mse}")
            i = i - 1
            regressor = regressor_copy
            xs = xs_backup
            ys = ys_backup
            continue

        # y_pred = regressor.predict(x_test_ae_cnn)
        # mse = mean_squared_error(y_true=y_test_ae_cnn, y_pred=y_pred)
        mse_dict[i] = mse
        print(f"Evaluating model at {i}th query...mse:{mse}")
        _chosen_x, _chosen_y = [], []
        xs, ys = [], []
    i += 1

Begin active learning process...
Initialize active learner...
28/28 [==============================] - 1s 40ms/step - loss: 30205.8906
active learning for 1700 epochs
